In [1]:
import os
import itertools
import multiprocessing as mp
from pathlib import Path

from concurrent.futures import ProcessPoolExecutor
from collections import defaultdict

import numpy as np
import pandas as pd
import supervision as sv

from trackers import OCSORTTracker

from trackers.eval import evaluate_mot_sequences


# Root directory for data.
# This notebook lives in `.../trackers metrics/mot17`, while
# `MOT17_yolox_dets` is also in `mot17/`.
DATA_ROOT = "."

MOT17_DET_ROOT = os.path.join(DATA_ROOT, "MOT17_yolox_dets")

# Optional GT root for TrackEval (if you have MOT17 GT downloaded in MOTChallenge format)
#MOT17_GT_ROOT = os.path.join("TrackEval", "data", "gt", "MOT17")
#MOT17_GT_TRAINVAL = os.path.join(MOT17_GT_ROOT, "train_val")


def get_yolox_detections(frame_id, det_list):
    dets_per_frame = [d for d in det_list if d.split(",")[0] == str(frame_id)]

    dets = []
    for line in dets_per_frame:
        det = line.split(",")  # "frame_no,x1,y1,x2,y2,det_conf"
        x1 = float(det[1])
        y1 = float(det[2])
        x2 = float(det[3])
        y2 = float(det[4])
        conf = float(det[5])
        dets.append([x1, y1, x2, y2, conf])
    return dets


def build_dets_index(det_list):
    dets_by_frame = defaultdict(list)
    for line in det_list:
        frame_id = int(line.split(",")[0])
        dets_by_frame[frame_id].append(line)
    return dets_by_frame


def get_detections_from_dict(frame_id, dets_by_frame):
    dets = []
    for line in dets_by_frame[frame_id]:
        det = line.split(",")
        x1 = float(det[1])
        y1 = float(det[2])
        x2 = float(det[3])
        y2 = float(det[4])
        conf = float(det[5])
        dets.append([x1, y1, x2, y2, conf])
    return dets


def write_mot_output(out_path: str):
    """Convert `MOT17-XX.txt` files into MOT17-server format.

    Creates three files per sequence: `MOT17-XX-{FRCNN,SDP,DPM}.txt`
    and deletes the unsuffixed original, matching the logic used in
    `trackers_ocsort_MOT17_runs.ipynb`.
    """
    out_dir = Path(out_path)

    existing = ["01", "03", "06", "07", "08", "12", "14"]
    missing = ["02", "04", "05", "09", "10", "11", "13"]
    suffixes = ["FRCNN", "SDP", "DPM"]

    # Replicate existing MOT17-XX.txt into 3 suffixed files, then delete original
    for num in existing:
        src = out_dir / f"MOT17-{num}.txt"
        if not src.exists():
            print(f"Missing expected source file: {src}")
            continue

        content = src.read_bytes()
        for suf in suffixes:
            dst = out_dir / f"MOT17-{num}-{suf}.txt"
            dst.write_bytes(content)

        src.unlink()
        print(f"Replicated + removed: {src.name}")

    # Create empty files for missing numbers
    for num in missing:
        for suf in suffixes:
            dst = out_dir / f"MOT17-{num}-{suf}.txt"
            dst.touch(exist_ok=True)

    print("Empty MOT17 server files created for missing sequences.")


In [3]:
def run_ocsort_mot17_once(
    split: str,
    lost_track_buffer: int,
    minimum_iou_threshold: float,
    minimum_consecutive_frames: int,
    direction_consistency_weight: float,
    high_conf_det_threshold: float,
    delta_t: int,
):
    """Run OC-SORT on one MOT17 split (train/val/test) for a given hyperparameter set.

    - For all splits: writes MOT-format `MOT17-XX*.txt` files under a
      unique directory in `OCSORT_outputs_MOT17/`.
    - If MOT17 GT is available under `MOT17_GT_ROOT` and `HAS_EVAL` is True,
      evaluates CLEAR, HOTA, and Identity metrics for non-test splits.
    - For the **test** split, also converts outputs to MOT17-server format
      (`MOT17-XX-FRCNN.txt`, etc.) via `write_mot_output`.
    """

    assert split in {"train", "val", "test"}, f"Unsupported split: {split}"

    tracker = OCSORTTracker(
        high_conf_det_threshold=high_conf_det_threshold,
        minimum_iou_threshold=minimum_iou_threshold,
        minimum_consecutive_frames=minimum_consecutive_frames,
        delta_t=delta_t,
        direction_consistency_weight=direction_consistency_weight,
        lost_track_buffer=lost_track_buffer,
    )

    det_root = os.path.join(MOT17_DET_ROOT, split)

    outputs_root = "OCSORT_outputs_MOT17"
    os.makedirs(outputs_root, exist_ok=True)

    save_dir = os.path.join(
        outputs_root,
        f"{split}_L{lost_track_buffer}_I{minimum_iou_threshold}__C{minimum_consecutive_frames}_"
        f"D{direction_consistency_weight}_H{high_conf_det_threshold}_T{delta_t}",
    )
    os.makedirs(save_dir, exist_ok=True)

    for seq in sorted(os.listdir(det_root)):
        if not seq.endswith(".txt"):
            continue

        tracker.reset()
        seq_name_raw = os.path.splitext(seq)[0]
        # For MOT17-val/train, rename to MOT17-XX-FRCNN so it matches GT + seqmap
        if split in {"train", "val"}:
            seq_name = seq_name_raw.split("_")[0] + "-FRCNN"
        else:
            seq_name = seq_name_raw

        with open(os.path.join(det_root, seq), "r") as f_det:
            det_list = f_det.readlines()

        last_frame = int(det_list[-1].split(",")[0])
        output_lines = []

        for frame_id in range(1, last_frame + 1):
            raw_dets = get_yolox_detections(frame_id, det_list)
            if raw_dets:
                raw_dets = np.array(raw_dets)
                dets = sv.Detections(
                    xyxy=raw_dets[:, :4],
                    confidence=raw_dets[:, 4],
                )
            else:
                dets = sv.Detections.empty()

            dets = tracker.update(detections=dets)

            for tid, (left, top, right, bottom) in zip(dets.tracker_id, dets.xyxy):
                if tid == -1:
                    continue

                width = right - left
                height = bottom - top

                output_lines.append(
                    f"{frame_id},{int(tid)},{round(left,1)},{round(top,1)},{round(width,1)},{round(height,1)},-1,-1,-1,-1\n"
                )

        save_path = os.path.join(save_dir, seq_name + ".txt")
        with open(save_path, "w") as f:
            f.writelines(output_lines)

        #print(f"Finished {split} sequence {seq_name} -> {save_path}")

    # For test split, create MOT17 server-compatible outputs
    if split == "test":
        write_mot_output(save_dir)

    # Try to evaluate with TrackEval on MOT17 **val-half** sequences only
    HOTA = IDF1 = MOTA = None
    if split == "val":
        # Notebook cwd is `mot17/`; val-half GT lives under this path
        gt_dir = "TrackEval/data/gt/MOT17_yolox_val/train_val"
        try:
            result = evaluate_mot_sequences(
                gt_dir=gt_dir,
                tracker_dir=save_dir,
                metrics=["CLEAR", "HOTA", "Identity"],
            )

            MOTA = result.to_dict()["aggregate"]["CLEAR"]["MOTA"]
            HOTA = result.to_dict()["aggregate"]["HOTA"]["HOTA"]
            IDF1 = result.to_dict()["aggregate"]["Identity"]["IDF1"]
            print(HOTA)
            print(
                f"split={split} | L:{lost_track_buffer}, I:{minimum_iou_threshold}, "
                f"C:{minimum_consecutive_frames}, D:{direction_consistency_weight}, "
                f"H:{high_conf_det_threshold}, T:{delta_t} -> "
                f"HOTA: {HOTA:.3f}, IDF1: {IDF1:.3f}, MOTA: {MOTA:.3f} (results in {save_dir})"
            )
        except Exception as e:
            print(
                f"split={split} | L:{lost_track_buffer}, I:{minimum_iou_threshold}, "
                f"C:{minimum_consecutive_frames}, D:{direction_consistency_weight}, "
                f"H:{high_conf_det_threshold}, T:{delta_t} -> "
                f"evaluation FAILED ({repr(e)}). Results saved in {save_dir}"
            )

    return {
        "split": split,
        "lost_track_buffer": lost_track_buffer,
        "minimum_iou_threshold": minimum_iou_threshold,
        "minimum_consecutive_frames": minimum_consecutive_frames,
        "direction_consistency_weight": direction_consistency_weight,
        "high_conf_det_threshold": high_conf_det_threshold,
        "delta_t": delta_t,
        "HOTA": HOTA,
        "IDF1": IDF1,
        "MOTA": MOTA,
        "output_dir": save_dir,
    }

In [4]:
# Define OC-SORT hyperparameter search space for MOT17 (similar to SoccerNet/SportsMOT)

lost_track_buffer_candidates = [10, 30, 60]
minimum_iou_threshold_candidates = [0.1, 0.3, 0.5]
minimum_consecutive_frames_candidates = [3, 5]
direction_consistency_weight_candidates = [0.0, 0.2, 0.5]
high_conf_det_threshold_candidates = [0.4, 0.6, 0.8]
delta_t_candidates = [1, 3]

all_candidate_lists = [
    lost_track_buffer_candidates,
    minimum_iou_threshold_candidates,
    minimum_consecutive_frames_candidates,
    direction_consistency_weight_candidates,
    high_conf_det_threshold_candidates,
    delta_t_candidates,
]

parameter_combinations = list(itertools.product(*all_candidate_lists))

print(f"Total OC-SORT parameter combinations for MOT17: {len(parameter_combinations)}")

Total OC-SORT parameter combinations for MOT17: 324


In [5]:
def tune_ocsort_on_mot17_val_parallel(parameter_combinations):
    """Run OC-SORT over MOT17 **val** for many hyperparameter sets in parallel.

    For each combination, writes MOT-format outputs and, if GT is available
    under `MOT17_GT_ROOT` and TrackEval is installed, also stores HOTA/IDF1/MOTA
    in the returned DataFrame and CSV.
    """

    num_workers = os.cpu_count()
    print(f"Running {len(parameter_combinations)} OC-SORT combinations on MOT17-val with {num_workers} workers")

    ctx = mp.get_context("fork")
    all_results = []

    with ProcessPoolExecutor(max_workers=num_workers, mp_context=ctx) as executor:
        futures = []
        for i, comb in enumerate(parameter_combinations):
            print(f"Submitting combination {i + 1}/{len(parameter_combinations)}")
            (
                lost_track_buffer,
                minimum_iou_threshold,
                minimum_consecutive_frames,
                direction_consistency_weight,
                high_conf_det_threshold,
                delta_t,
            ) = comb
            futures.append(
                executor.submit(
                    run_ocsort_mot17_once,
                    "val",
                    lost_track_buffer,
                    minimum_iou_threshold,
                    minimum_consecutive_frames,
                    direction_consistency_weight,
                    high_conf_det_threshold,
                    delta_t,
                )
            )

        for i, f in enumerate(futures):
            try:
                result = f.result()
                all_results.append(result)
                print(f"[{i + 1}/{len(parameter_combinations)}] val combination finished.")
            except Exception as e:
                print(f"[{i + 1}/{len(parameter_combinations)}] val combination FAILED: {repr(e)}")

    df = pd.DataFrame(all_results)
    print("Validation hyperparameter sweep complete. Results stored in 'ocsort_MOT17_val_tuning_df'.")
    print(df.head())

    if not df.empty:
        df.to_csv("ocsort_MOT17_val_tuning.csv", index=False)
    else:
        print("Warning: no successful validation runs. Check paths and errors printed above.")

    return df


# 1) Generate OC-SORT outputs (and metrics if possible) for all combinations on MOT17-val
ocsort_MOT17_val_tuning_df = tune_ocsort_on_mot17_val_parallel(parameter_combinations)

# 2) If HOTA is available, pick best params by HOTA
if not ocsort_MOT17_val_tuning_df.empty and "HOTA" in ocsort_MOT17_val_tuning_df.columns and not ocsort_MOT17_val_tuning_df["HOTA"].isna().all():
    best_idx = ocsort_MOT17_val_tuning_df["HOTA"].idxmax()
    best_row = ocsort_MOT17_val_tuning_df.loc[best_idx]
    print("\nBest validation row by HOTA:\n", best_row)

    best_params = dict(
        lost_track_buffer=int(best_row["lost_track_buffer"]),
        minimum_iou_threshold=float(best_row["minimum_iou_threshold"]),
        minimum_consecutive_frames=int(best_row["minimum_consecutive_frames"]),
        direction_consistency_weight=float(best_row["direction_consistency_weight"]),
        high_conf_det_threshold=float(best_row["high_conf_det_threshold"]),
        delta_t=int(best_row["delta_t"]),
    )
    print("\nBest hyperparameters (val):", best_params)
else:
    print("\nNo HOTA column found in 'ocsort_MOT17_val_tuning_df' – you can still inspect 'output_dir' per row and evaluate externally.")

Running 324 OC-SORT combinations on MOT17-val with 10 workers
Submitting combination 1/324
Submitting combination 2/324
Submitting combination 3/324
Submitting combination 4/324
Submitting combination 5/324
Submitting combination 6/324
Submitting combination 7/324
Submitting combination 8/324
Submitting combination 9/324
Submitting combination 10/324
Submitting combination 11/324
Submitting combination 12/324
Submitting combination 13/324
Submitting combination 14/324
Submitting combination 15/324
Submitting combination 16/324
Submitting combination 17/324
Submitting combination 18/324
Submitting combination 19/324
Submitting combination 20/324
Submitting combination 21/324
Submitting combination 22/324
Submitting combination 23/324
Submitting combination 24/324
Submitting combination 25/324
Submitting combination 26/324
Submitting combination 27/324
Submitting combination 28/324
Submitting combination 29/324
Submitting combination 30/324
Submitting combination 31/324
Submitting combin

## Final MOT17-test run and submission archive

After you have chosen the best hyperparameters (either automatically via HOTA
if GT is available, or manually by inspecting results / evaluation server),
run the cell below to:

- Run OC-SORT on **MOT17-test** with the selected parameters.
- Convert results into MOT17 server format (`MOT17-XX-FRCNN.txt`, etc.).
- Create a **zip archive** ready for submission to the MOTChallenge server.

In [7]:
ocsort_MOT17_val_tuning_df = pd.read_csv("ocsort_MOT17_val_tuning.csv")
best_idx = ocsort_MOT17_val_tuning_df["HOTA"].idxmax()
best_row = ocsort_MOT17_val_tuning_df.loc[best_idx]
print("\nBest validation row by HOTA:\n", best_row)

best_params = dict(
    lost_track_buffer=int(best_row["lost_track_buffer"]),
    minimum_iou_threshold=float(best_row["minimum_iou_threshold"]),
    minimum_consecutive_frames=int(best_row["minimum_consecutive_frames"]),
    direction_consistency_weight=float(best_row["direction_consistency_weight"]),
    high_conf_det_threshold=float(best_row["high_conf_det_threshold"]),
    delta_t=int(best_row["delta_t"]),
)
print("\nBest hyperparameters (val):", best_params)


Best validation row by HOTA:
 split                                                                         val
lost_track_buffer                                                              30
minimum_iou_threshold                                                         0.3
minimum_consecutive_frames                                                      3
direction_consistency_weight                                                  0.2
high_conf_det_threshold                                                       0.4
delta_t                                                                         1
HOTA                                                                     0.503155
IDF1                                                                     0.543643
MOTA                                                                     0.422964
output_dir                      OCSORT_outputs_MOT17/val_L30_I0.3__C3_D0.2_H0....
Name: 150, dtype: object

Best hyperparameters (val): {'lost_track_

In [8]:
import shutil


print("Using MOT17-test hyperparameters:", best_params)

# Run on test split with best hyperparameters
test_result = run_ocsort_mot17_once(
    split="test",
    **best_params,
)

print("\nTest split metrics (if GT available):")
print({k: test_result.get(k) for k in ["HOTA", "IDF1", "MOTA"]})

# Create a zip archive of the test outputs for submission
submission_dir = test_result["output_dir"]
submission_zip_base = os.path.basename(submission_dir)
submission_zip = submission_zip_base + ".zip"

shutil.make_archive("oc_sort"+submission_zip_base, "zip", submission_dir)
print("\nCreated MOT17 submission archive:", submission_zip)
print("Archive contains MOT17 server-format txt files ready for upload.")

Using MOT17-test hyperparameters: {'lost_track_buffer': 30, 'minimum_iou_threshold': 0.3, 'minimum_consecutive_frames': 3, 'direction_consistency_weight': 0.2, 'high_conf_det_threshold': 0.4, 'delta_t': 1}


Replicated + removed: MOT17-01.txt
Replicated + removed: MOT17-03.txt
Replicated + removed: MOT17-06.txt
Replicated + removed: MOT17-07.txt
Replicated + removed: MOT17-08.txt
Replicated + removed: MOT17-12.txt
Replicated + removed: MOT17-14.txt
Empty MOT17 server files created for missing sequences.

Test split metrics (if GT available):
{'HOTA': None, 'IDF1': None, 'MOTA': None}

Created MOT17 submission archive: test_L30_I0.3__C3_D0.2_H0.4_T1.zip
Archive contains MOT17 server-format txt files ready for upload.
